# 08 - Preprocessing Summary

This notebook summarizes the data-processing pipeline for DATATHON 2026. It is intentionally documentation-first: raw files in `dataset/` are not modified.

## Data Understanding

- Load raw CSV files and parse date columns.
- Check shape, columns, sample rows, primary keys, foreign keys, missing values, and date ranges.
- Outputs live in `artifacts/data_understanding/`.

## Cleaning

Cleaning is deliberately conservative. Not every null should be filled.

- `order_items.promo_id` null means no primary promotion.
- `order_items.promo_id_2` null means no stacked or second promotion.
- `promotions.applicable_category` null likely means broad or all-category promotion metadata.

The project records these meanings in `artifacts/data_understanding/missing_values.csv`.

## Preprocessing

Preprocessing converts raw relational tables into analysis-ready grains:

- `aggregated_tables/order_lines_enriched.csv`: one row per order item line.
- `aggregated_tables/returns_enriched.csv`: one row per return record.
- `aggregated_tables/inventory_monthly_enriched.csv`: one row per month, category, segment.
- `aggregated_tables/table_grain_map.csv`: grain documentation.

These outputs support MCQ and EDA without changing raw input files.

## Forecasting Feature Engineering

Forecasting uses one row per date, matching `sales.csv` and `sample_submission.csv`.

Feature groups:

- Calendar features.
- Revenue and COGS lag features.
- Rolling mean and rolling standard deviation features.
- Exponentially weighted Revenue features.
- Promotion calendar features when they are available from supplied data.

Main outputs:

- `aggregated_tables/daily_sales_features_final_grain.csv`
- `artifacts/final_submission/final_feature_list.csv`

## Model Preprocessing

- Numeric imputation uses `SimpleImputer(strategy="median")` inside sklearn pipelines.
- Scaling is applied only to linear/ridge candidates in notebook 06.
- Tree-based and XGBoost models use imputation but do not need scaling.

## Leakage Controls

- `sample_submission` Revenue and COGS are ignored.
- Sales-derived lag and rolling features use past values only.
- Future predictions are generated recursively.
- Recursive walk-forward validation is exported to `artifacts/final_submission/recursive_backtest_2022_scores.csv`.

In [ ]:
import pandas as pd

summary = pd.read_csv('artifacts/preprocessing/preprocessing_summary.csv')
summary